In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
import os
from typing import List
from typing import Sequence
from typing import Tuple

from torchvision.datasets import Omniglot
from torchvision import transforms
from torch.utils.data import DataLoader

import torch 
import numpy as np
import matplotlib.pyplot as plt


In [3]:
print(torch.cuda.is_available())

True


In [ ]:
from matplotlib.gridspec import GridSpec
from src.utils import compute_kl_g, compute_re_g
from sklearn.decomposition import PCA

# Generate example distribution parameters for multivariate diagonal case
def generate_examples(num_examples=4, dim=100):
    examples = {}
    
    for i in range(1, num_examples + 1):
        # Create different scenarios for means and variances
        if i == 1:
            # Similar distributions
            mean = torch.randn(dim) * 0.5
            prior_mean = mean + torch.randn(dim) * 0.2
            log_var = torch.randn(dim) * 0.3 - 1.0  # Slightly negative to get small variances
            log_prior_var = log_var + torch.randn(dim) * 0.2
        elif i == 2:
            # Very different means
            mean = torch.randn(dim) * 2.0
            prior_mean = -mean + torch.randn(dim) * 0.5
            log_var = torch.randn(dim) * 0.3 - 0.5
            log_prior_var = torch.randn(dim) * 0.3 - 0.5
        elif i == 3:
            # Very different variances
            mean = torch.randn(dim) * 0.5
            prior_mean = mean + torch.randn(dim) * 0.3
            log_var = torch.randn(dim) * 0.3 - 2.0  # Small variances
            log_prior_var = torch.randn(dim) * 0.3 + 1.0  # Large variances
        else:
            # Both different means and variances
            mean = torch.randn(dim) * 1.5
            prior_mean = -mean + torch.randn(dim) * 1.0
            log_var = torch.randn(dim) * 0.5
            log_prior_var = -log_var + torch.randn(dim) * 0.5
        
        examples[f"example_{i}"] = {
            "mean": mean,
            "log_var": log_var,
            "prior_mean": prior_mean,
            "log_prior_var": log_prior_var,
            "description": {
                1: "Similar distributions",
                2: "Different means",
                3: "Different variances",
                4: "Different means and variances"
            }[i]
        }
    
    return examples

# Function to compute divergences for different alpha values
def compute_divergences(examples, alpha_values, q_value=1.0, v_value=1.0, lamb_value=1.0, initial_prior_var=1.0):
    results = {}
    
    for ex_name, ex_data in examples.items():
        mean = ex_data["mean"]
        log_var = ex_data["log_var"]
        prior_mean = ex_data["prior_mean"]
        log_prior_var = ex_data["log_prior_var"]
        
        kl_value, _ , _ = compute_kl_g(
            mean, log_var, prior_mean, log_prior_var, 
            q=q_value, v=v_value, sum=True, lamb=lamb_value, 
            initial_prior_var=initial_prior_var
        )
        kl_value = kl_value.item()
        
        re_values = []
        for alpha in alpha_values:
            if abs(alpha - 1.0) < 1e-6:
                # KL divergence is Rényi divergence when alpha → 1
                re_values.append(kl_value)
            else:
                re_value, _, _ = compute_re_g(
                    mean, log_var, prior_mean, log_prior_var, 
                    alpha=alpha, v=v_value, sum=True, lamb=lamb_value, 
                    initial_prior_var=initial_prior_var
                )
                re_value = re_value.item()
                re_values.append(re_value)
        
        results[ex_name] = {
            "kl": kl_value,
            "re": dict(zip(alpha_values, re_values)),
            "description": ex_data["description"]
        }
    
    return results

# Function to visualize the results
def plot_results(results, alpha_values):
    fig = plt.figure(figsize=(15, 12))
    gs = GridSpec(2, 2, figure=fig)
    
    # Plot each example
    for i, (ex_name, ex_data) in enumerate(results.items()):
        row, col = i // 2, i % 2
        ax = fig.add_subplot(gs[row, col])
        
        kl_value = ex_data["kl"]
        re_values = [ex_data["re"][alpha] for alpha in alpha_values]
        
        # Plot KL as a horizontal line
        ax.axhline(y=kl_value, color='r', linestyle='-', label='KL Divergence')
        
        # Plot Rényi divergence for different alpha values
        ax.plot(alpha_values, re_values, 'bo-', label='Rényi Divergence')
        
        # Add labels and title
        ax.set_xlabel('Alpha (α)')
        ax.set_ylabel('Divergence Value')
        ax.set_title(f"Example {i+1}: {ex_data['description']}")
        ax.grid(True, linestyle='--', alpha=0.7)
        ax.legend()
        
        # Annotate the point where alpha=1
        idx_alpha_1 = [i for i, a in enumerate(alpha_values) if abs(a - 1.0) < 1e-6]
        if idx_alpha_1:
            idx = idx_alpha_1[0]
            ax.plot(1.0, re_values[idx], 'ro', markersize=8)
            ax.annotate(f'α=1 (KL)', 
                        xy=(1.0, re_values[idx]),
                        xytext=(1.0 + 0.1, re_values[idx] + 0.5),
                        arrowprops=dict(facecolor='black', shrink=0.05, width=1.5))
    
    plt.tight_layout()
    return fig

'''
# Function to create difference plot
def plot_difference(results, alpha_values):
    fig, ax = plt.subplots(figsize=(10, 6))
    
    for ex_name, ex_data in results.items():
        kl_value = ex_data["kl"]
        re_values = [ex_data["re"][alpha] for alpha in alpha_values]
        differences = [re - kl_value for re in re_values]
        
        ax.plot(alpha_values, differences, 'o-', label=f"{ex_name}: {ex_data['description']}")
    
    ax.axhline(y=0, color='k', linestyle='--', alpha=0.7)
    ax.set_xlabel('Alpha (α)')
    ax.set_ylabel('Rényi - KL Difference')
    ax.set_title('Difference Between Rényi and KL Divergences')
    ax.grid(True, linestyle='--', alpha=0.7)
    ax.legend()
    
    plt.tight_layout()
    return fig
'''
# Function to visualize diagonal variances
def plot_variances(examples, dim=3):
    num_examples = len(examples)
    fig, axes = plt.subplots(num_examples, 1, figsize=(10, 3*num_examples))
    
    for i, (ex_name, ex_data) in enumerate(examples.items()):
        # Convert log variances to variances
        variance = torch.exp(ex_data["log_var"]).detach().numpy()
        prior_variance = torch.exp(ex_data["log_prior_var"]).detach().numpy()
        
        # Set up the bar positions
        x = np.arange(dim)
        width = 0.35
        
        # Create the bars
        axes[i].bar(x - width/2, variance, width, label='Posterior Variance')
        axes[i].bar(x + width/2, prior_variance, width, label='Prior Variance')
        
        # Add labels and title
        axes[i].set_title(f'Example {i+1}: {ex_data["description"]}')
        axes[i].set_xlabel('Dimension')
        axes[i].set_ylabel('Variance')
        axes[i].set_xticks(x)
        axes[i].legend()
        
        # Add mean information as text
        mean_diff_norm = torch.norm(ex_data["mean"] - ex_data["prior_mean"]).item()
        axes[i].text(0.02, 0.9, f'Mean diff norm: {mean_diff_norm:.4f}', 
                  transform=axes[i].transAxes, fontsize=10)
    
    plt.tight_layout()
    return fig

# Function to create difference plot
def plot_difference(results, alpha_values):
    fig, ax = plt.subplots(figsize=(10, 6))
    
    for ex_name, ex_data in results.items():
        kl_value = ex_data["kl"]
        re_values = [ex_data["re"][alpha] for alpha in alpha_values]
        differences = [re - kl_value for re in re_values]
        
        ax.plot(alpha_values, differences, 'o-', label=f"{ex_name}: {ex_data['description']}")
    
    ax.axhline(y=0, color='k', linestyle='--', alpha=0.7)
    ax.set_xlabel('Alpha (α)')
    ax.set_ylabel('Rényi - KL Difference')
    ax.set_title('Difference Between Rényi and KL Divergences')
    ax.grid(True, linestyle='--', alpha=0.7)
    ax.legend()
    
    plt.tight_layout()
    return fig

# Function to visualize variances and means using PCA
def plot_variances_pca(examples, n_components=2):
    num_examples = len(examples)
    fig, axes = plt.subplots(num_examples, 1, figsize=(10, 3*num_examples))
    
    # Collect all means and variances
    all_means = []
    all_prior_means = []
    all_variances = []
    all_prior_variances = []
    
    for ex_name, ex_data in examples.items():
        all_means.append(ex_data["mean"].numpy())
        all_prior_means.append(ex_data["prior_mean"].numpy())
        all_variances.append(torch.exp(ex_data["log_var"]).detach().numpy())
        all_prior_variances.append(torch.exp(ex_data["log_prior_var"]).detach().numpy())
    
    # Convert to numpy arrays
    all_means = np.array(all_means)
    all_prior_means = np.array(all_prior_means)
    all_variances = np.array(all_variances)
    all_prior_variances = np.array(all_prior_variances)
    
    # Apply PCA
    pca = PCA(n_components=n_components)
    means_pca = pca.fit_transform(all_means)
    prior_means_pca = pca.transform(all_prior_means)
    variances_pca = pca.fit_transform(all_variances)
    prior_variances_pca = pca.transform(all_prior_variances)
    
    for i, (ex_name, ex_data) in enumerate(examples.items()):
        # Plot PCA results
        axes[i].bar(np.arange(n_components) - 0.2, variances_pca[i], 0.4, label='Posterior Variance (PCA)')
        axes[i].bar(np.arange(n_components) + 0.2, prior_variances_pca[i], 0.4, label='Prior Variance (PCA)')
        
        # Add labels and title
        axes[i].set_title(f'Example {i+1}: {ex_data["description"]}')
        axes[i].set_xlabel('PCA Component')
        axes[i].set_ylabel('Variance')
        axes[i].legend()
        
        # Add mean information as text
        mean_diff_norm = torch.norm(ex_data["mean"] - ex_data["prior_mean"]).item()
        axes[i].text(0.02, 0.9, f'Mean diff norm: {mean_diff_norm:.4f}', 
                  transform=axes[i].transAxes, fontsize=10)
    
    plt.tight_layout()
    return fig

# Main execution
# Define alpha values to test (including 1.0 for KL comparison)
alpha_values = [0.1, 0.25, 0.5, 0.75, 0.9, 0.99, 1.001, 1.01, 1.1, 1.5, 2.0, 3.0, 5.0]

# Set dimensionality
dim = 100

# Generate examples
examples = generate_examples(num_examples=4, dim=dim)

# Display the generated examples
print("Generated Examples:")
for ex_name, ex_data in examples.items():
    print(f"\n{ex_name}: {ex_data['description']}")
    print(f"Mean: {ex_data['mean']}")
    print(f"Prior Mean: {ex_data['prior_mean']}")
    print(f"Log Variance: {ex_data['log_var']}")
    print(f"Log Prior Variance: {ex_data['log_prior_var']}")

# Compute divergences
results = compute_divergences(examples, alpha_values)

# Plot the results
divergence_fig = plot_results(results, alpha_values)
difference_fig = plot_difference(results, alpha_values)
variance_fig = plot_variances_pca(examples, n_components=2)

# Print the numerical results
print("\nNumerical Results:")
for ex_name, ex_data in results.items():
    print(f"\n{ex_name}: {ex_data['description']}")
    print(f"KL Divergence: {ex_data['kl']:.4f}")
    print("Rényi Divergences:")
    for alpha, value in ex_data["re"].items():
        print(f"  α={alpha:.2f}: {value:.4f}" + (" (KL)" if abs(alpha - 1.0) < 1e-6 else ""))

# Save the figures
#divergence_fig.savefig('diagonal_divergence_comparison.png')
#difference_fig.savefig('diagonal_divergence_difference.png')
#variance_fig.savefig('diagonal_variances.png')

# Create a plot to show behavior of Renyi divergence around alpha=1
# This helps to verify the limit behavior
def plot_alpha_near_one(examples, lamb_value=1.0, initial_prior_var=1.0):
    # Define fine-grained alpha values near 1
    alpha_fine = np.concatenate([
        np.linspace(0.9, 0.999, 20),
        np.array([0.9999, 0.99999, 1.0, 1.00001, 1.0001]),
        np.linspace(1.001, 1.1, 20)
    ])
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    axes = axes.flatten()
    
    for i, (ex_name, ex_data) in enumerate(examples.items()):
        mean = ex_data["mean"]
        log_var = ex_data["log_var"]
        prior_mean = ex_data["prior_mean"]
        log_prior_var = ex_data["log_prior_var"]
        
        # Calculate KL once
        kl_value, _, _ = compute_kl_g(
            mean, log_var, prior_mean, log_prior_var, 
            q=1.0, v=1.0, sum=True, lamb=lamb_value, 
            initial_prior_var=initial_prior_var
        )
        kl_value = kl_value.item()
        
        # Calculate Renyi for fine alpha values
        re_values = []
        for alpha in alpha_fine:
            if abs(alpha - 1.0) < 1e-10:
                re_values.append(kl_value)
            else:
                re_value, _, _ = compute_re_g(
                    mean, log_var, prior_mean, log_prior_var, 
                    alpha=alpha, v=1.0, sum=True, lamb=lamb_value, 
                    initial_prior_var=initial_prior_var
                )
                re_value = re_value.item()
                re_values.append(re_value)
        
        # Plot
        axes[i].plot(alpha_fine, re_values, 'bo-')
        axes[i].axhline(y=kl_value, color='r', linestyle='-', label='KL Divergence')
        axes[i].axvline(x=1.0, color='g', linestyle='--')
        
        # Highlight point at alpha=1
        idx_alpha_1 = np.where(np.abs(alpha_fine - 1.0) < 1e-10)[0][0]
        axes[i].plot(1.0, re_values[idx_alpha_1], 'ro', markersize=8)
        
        # Add labels
        axes[i].set_xlabel('Alpha (α)')
        axes[i].set_ylabel('Divergence Value')
        axes[i].set_title(f"Example {i+1}: {ex_data['description']}")
        axes[i].grid(True, linestyle='--', alpha=0.7)
        axes[i].legend()
        
        # Set x-axis limits to zoom in around alpha=1
        axes[i].set_xlim(0.95, 1.05)
    
    plt.tight_layout()
    return fig

# Plot behavior around alpha=1
alpha_near_one_fig = plot_alpha_near_one(examples)
#alpha_near_one_fig.savefig('alpha_near_one.png')

# In a typical environment, these would display
divergence_fig.show()
difference_fig.show()
variance_fig.show()
alpha_near_one_fig.show()

In [ ]:
examples["example_1"]["mean"].shape

for key, example in examples.items():
    print(key)
    print(example["mean"].shape)
    #print(example["log_var"].shape)
    #print(example["prior_mean"].shape)
   # print(example["log_prior_var"].shape)

In [ ]:
import matplotlib.pyplot as plt
# Function to plot Gaussian distributions
def plot_examples(examples):
    plt.figure(figsize=(12, 8))
    
    for key, example in examples.items():
        mean = example['mean'].numpy()
        log_var = example['log_var'].numpy()
        prior_mean = example['prior_mean'].numpy()
        log_prior_var = example['log_prior_var'].numpy()
        
        cov = np.diag(np.exp(log_var))
        prior_cov = np.diag(np.exp(log_prior_var))
        
        # Generate samples
        samples = np.random.multivariate_normal(mean, cov, 500)
        prior_samples = np.random.multivariate_normal(prior_mean, prior_cov, 500)
        
        plt.scatter(samples[:, 0], samples[:, 1], alpha=0.5, label=f'{key} - {example["description"]} (mean)')
        plt.scatter(prior_samples[:, 0], prior_samples[:, 1], alpha=0.5, label=f'{key} - {example["description"]} (prior mean)')
    
    plt.xlabel('X-axis')
    plt.ylabel('Y-axis')
    plt.title('Multivariate Gaussian Distributions')
    plt.legend()
    plt.grid(True)
    plt.show()

# Generate examples and plot them
plot_examples(examples)

## Run test for VCL that compares Renyi and KL values

In [ ]:
import subprocess

# Define the parameters
experiment = "cifar"
approach = "vcl"
reg_type = "re_g"
dof = 50
q = [0.5, 0.7, 1.01, 2.2]
seed = [93, 187]

# Loop over the seeds and run the experiment
for alpha in q:
    for sd in seed:
        print(f"Running experiment with alpha {alpha} and seed {sd}")
        subprocess.run([
            "python", "./src/run.py",
            "--experiment", experiment,
            "--approach", approach,
            "--seed", str(sd),
            "--use-sweep", str(True),
            "--reg_type", reg_type,
            "--q", str(alpha),
        ])

## Run omniglot

In [ ]:
import subprocess
# Define the parameters
experiment = "omniglot"
approach = "vcl"
reg_type = "re_g"
dof = 50
seed = [14]
#q = [0.5, 0.7, 0.999, 1.001, 1.01, 1.5, 2.5]

# Loop over the seeds and run the experiment

for sd in seed:
    print(f"Running experiment with and alpha {alpha}")
    subprocess.run([
        "python", "./src/run.py",
        "--experiment", experiment,
        "--approach", approach,
        "--seed", str(sd),
        "--use-sweep", str(True),
        "--reg_type", reg_type,
    ])

In [5]:
import subprocess
# Define the parameters
experiment = "omniglot"
approach = "vcl"
reg_type = "re_g"
dof = 50
seed = 14
q = [1.5]

# Loop over the seeds and run the experiment

for alpha in q:
    print(f"Running experiment with and alpha {alpha}")
    subprocess.run([
        "python", "./src/run.py",
        "--experiment", experiment,
        "--approach", approach,
        "--seed", str(seed),
        "--q", str(alpha),
        "--use-sweep", str(True),
        "--reg_type", reg_type,
    ])

Running experiment with and alpha 1.5


/home/nelosegui/BIFOLD_work/CL_curved/gvcl_fork/src/dataloaders/omniglot.py:178: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data[i][s]['x']=torch.load(os.path.join(binari

Arguments =
	sweep_name: None
	seed: 14
	experiment: omniglot
	approach: vcl
	reg_type: re_g
	q: 1.5
	v: 1
	output: ./res/omniglot/vcl/re_g/1.5/14.txt
	nepochs: None
	lr: -1
	parameter: 
	ntasks: -1
	context: None
	momentum: 0.9
	weight_decay: 0.0001
	beta: 1
	lamb: 1
	root_path: ./
	use_best_hyperparams: False
	use_sweep: True
	train_samples: 10
	lr_schedule: False
	scheduler_type: cosine_anneal
	sbatch: 64
	optimizer: sgd
approach VCL True
using vcl model smnist
Load data...
Files already downloaded and verified
Files already downloaded and verified
Tasks permuted with seed 42
Task permutation: [42, 17, 30, 34, 20, 25, 0, 8, 36, 18, 28, 39, 3, 2, 41, 1, 37, 6, 14, 12, 11, 32, 43, 16, 5, 38, 31, 44, 26, 45, 15, 47, 4, 10, 21, 19, 33, 27, 13, 48, 46, 23, 22, 35, 7, 24, 49, 29, 40, 9]
Input size = (28, 28) 
Task info = [(0, 26), (1, 23), (2, 26), (3, 26), (4, 40), (5, 14), (6, 55), (7, 24), (8, 45), (9, 26), (10, 17), (11, 26), (12, 41), (13, 40), (14, 47), (15, 40), (16, 45), (17, 16),